# Notebook 04 — Multi-Seed Statistical Evaluation (M0 vs M2)



In [28]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import random
import time
from dataclasses import dataclass, field
from functools import partial
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image
from scipy import stats

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE = (torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cuda") if torch.cuda.is_available()
          else torch.device("cpu"))
print(f"Using device: {DEVICE} | torch {torch.__version__}")

Using device: mps | torch 2.12.0


In [29]:
@dataclass
class Config:
    data_root: Path = Path("../data/pharmacy_lk")
    train_csv: str = "splits/train.csv"
    val_csv: str = "splits/val.csv"
    test_csv: str = "splits/test.csv"
    img_dir: str = "images"
    img_col: str = "image_filename"
    label_col: str = "medicine_name"

    img_height: int = 48
    img_width: int = 320

    rnn_hidden: int = 256
    rnn_layers: int = 2
    dropout: float = 0.2

    batch_size: int = 64
    epochs: int = 120
    lr: float = 1e-4
    warmup_epochs: int = 5
    weight_decay: float = 1e-4
    grad_clip: float = 5.0
    early_stop_patience: int = 20
    num_workers: int = 0

    seeds: tuple = (42, 43, 44, 45, 46)        # 5 seeds
    best_tau: float = 0.6                       # from the M2 τ-sweep; fixed here for comparability
    formulary_path: "Path | None" = None
    out_dir: Path = Path("../checkpoints/multiseed")

CFG = Config()
CFG.out_dir.mkdir(parents=True, exist_ok=True)

## 1. Shared components (identical to baseline / M2)

In [30]:
class Vocab:
    BLANK = 0
    def __init__(self, texts):
        chars = sorted(set("".join(texts)))
        self.idx2char = {i + 1: c for i, c in enumerate(chars)}
        self.char2idx = {c: i for i, c in self.idx2char.items()}
    def __len__(self):
        return len(self.idx2char) + 1
    def encode(self, t):
        return [self.char2idx[c] for c in t]
    def decode_greedy(self, indices):
        out, prev = [], None
        for i in indices:
            if i != prev and i != self.BLANK:
                out.append(self.idx2char[i])
            prev = i
        return "".join(out)


class WordImageDataset(Dataset):
    def __init__(self, csv_path, img_dir, cfg, vocab=None, augment=False):
        self.df = pd.read_csv(csv_path).dropna(subset=[cfg.label_col])
        self.df[cfg.label_col] = self.df[cfg.label_col].astype(str).str.strip()
        self.img_dir = Path(img_dir); self.cfg = cfg; self.vocab = vocab; self.augment = augment
    def labels(self):
        return self.df[self.cfg.label_col].tolist()
    def __len__(self):
        return len(self.df)
    def _load(self, path):
        img = Image.open(path).convert("L"); w, h = img.size
        new_w = min(max(1, int(round(w * self.cfg.img_height / h))), self.cfg.img_width)
        img = img.resize((new_w, self.cfg.img_height), Image.BILINEAR)
        canvas = Image.new("L", (self.cfg.img_width, self.cfg.img_height), color=255)
        canvas.paste(img, (0, 0)); return canvas
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load(self.img_dir / str(row[self.cfg.img_col]))
        if self.augment:
            img = img.rotate(random.uniform(-3, 3), resample=Image.BILINEAR, fillcolor=255)
        x = torch.from_numpy(np.array(img, dtype=np.float32) / 255.0).unsqueeze(0)
        target = torch.tensor(self.vocab.encode(row[self.cfg.label_col]), dtype=torch.long)
        return x, target, row[self.cfg.label_col], str(row[self.cfg.img_col])


def collate(batch):
    xs, targets, texts, fnames = zip(*batch)
    tl = torch.tensor([t.numel() for t in targets], dtype=torch.long)
    return torch.stack(xs), torch.cat(targets), tl, list(texts), list(fnames)


def edit_distance(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return prev[-1]


def corpus_metrics(preds, refs):
    total_ed = sum(edit_distance(p, r) for p, r in zip(preds, refs))
    total_chars = sum(len(r) for r in refs)
    exact = sum(p == r for p, r in zip(preds, refs))
    return {"CER": total_ed / max(total_chars, 1), "ExactMatch": exact / len(refs), "n": len(refs)}


class CRNN(nn.Module):
    def __init__(self, n_classes, rnn_hidden=256, rnn_layers=2, dropout=0.2):
        super().__init__()
        def conv(i, o, bn=False):
            L = [nn.Conv2d(i, o, 3, 1, 1)]
            if bn: L.append(nn.BatchNorm2d(o))
            L.append(nn.ReLU(inplace=True)); return L
        self.cnn = nn.Sequential(
            *conv(1, 64), nn.MaxPool2d(2, 2),
            *conv(64, 128), nn.MaxPool2d(2, 2),
            *conv(128, 256), *conv(256, 256), nn.MaxPool2d((2, 1), (2, 1)),
            *conv(256, 512, bn=True), *conv(512, 512, bn=True), nn.MaxPool2d((2, 1), (2, 1)),
        )
        self.collapse = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(512, rnn_hidden, rnn_layers, bidirectional=True,
                           dropout=dropout if rnn_layers > 1 else 0.0)
        self.head = nn.Linear(2 * rnn_hidden, n_classes)
    def forward(self, x):
        f = self.collapse(self.cnn(x)).squeeze(2).permute(2, 0, 1)
        seq, _ = self.rnn(f)
        return self.head(seq)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

## 2. Lexicon (built once — training split only)

In [31]:
train_df = pd.read_csv(CFG.data_root / CFG.train_csv)
lexicon = set(train_df[CFG.label_col].astype(str).str.strip().str.lower())
if CFG.formulary_path and Path(CFG.formulary_path).exists():
    extra = {l.strip().lower() for l in open(CFG.formulary_path) if l.strip()}
    lexicon |= extra
    print(f"external formulary added: +{len(extra)}")
lexicon = sorted(lexicon)
lex_by_len = defaultdict(list)
for w in lexicon:
    lex_by_len[len(w)].append(w)
print(f"lexicon size: {len(lexicon)}")


def nearest_lexicon(word, max_len_gap=3):
    if not word:
        return None, 10**9
    if word in lex_by_len.get(len(word), ()):
        return word, 0
    best, best_d = None, 10**9
    for L in range(len(word) - max_len_gap, len(word) + max_len_gap + 1):
        for entry in lex_by_len.get(L, ()):
            d = edit_distance(word, entry)
            if d < best_d:
                best, best_d = entry, d
                if d == 1:
                    return best, best_d
    return best, best_d


def lexicon_decode(raw, tau):
    out = []
    for p in raw:
        entry, d = nearest_lexicon(p)
        out.append(entry if (entry is not None and d / max(len(p), 1) <= tau) else p)
    return out

lexicon size: 1210


## 3. Train + evaluate one seed
Returns per-example predictions (for both M0 and M2) plus metrics, so the significance
test downstream can work on matched test items.

In [32]:
def greedy_preds(model, loader, vocab):
    model.eval(); preds, refs, files = [], [], []
    with torch.no_grad():
        for xb, _, _, texts, fnames in loader:
            logits = model(xb.to(DEVICE))
            idx = logits.argmax(-1).permute(1, 0).cpu()
            preds += [vocab.decode_greedy(s.tolist()) for s in idx]
            refs += texts; files += fnames
    return preds, refs, files


def run_one_seed(seed, vocab, train_dl, val_dl, test_dl, seen_map):
    set_seed(seed)
    model = CRNN(len(vocab), CFG.rnn_hidden, CFG.rnn_layers, CFG.dropout).to(DEVICE)
    ctc = nn.CTCLoss(blank=Vocab.BLANK, zero_infinity=True)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=4)

    best_cer, no_improve, best_state = float("inf"), 0, None
    for epoch in range(1, CFG.epochs + 1):
        ep_t0 = time.time()
        if epoch <= CFG.warmup_epochs:
            for g in opt.param_groups:
                g["lr"] = CFG.lr * epoch / CFG.warmup_epochs
        model.train()
        for xb, targets, tlens, _, _ in train_dl:
            xb = xb.to(DEVICE)
            logits = model(xb); T = logits.shape[0]
            loss = ctc(logits.log_softmax(2).cpu(), targets,
                       torch.full((xb.shape[0],), T, dtype=torch.long), tlens)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip); opt.step()

        vp, vr, _ = greedy_preds(model, val_dl, vocab)
        vcer = corpus_metrics(vp, vr)["CER"]
        if epoch > CFG.warmup_epochs:
            sched.step(vcer)
        if vcer < best_cer:
            best_cer, no_improve = vcer, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= CFG.early_stop_patience:
                print(f"      [seed {seed}] early stop at epoch {epoch} "
                      f"(best val CER {best_cer:.4f})")
                break
        # heartbeat every 10 epochs so you can see progress + per-epoch time within a seed
        if epoch % 10 == 0 or epoch == 1:
            print(f"      [seed {seed}] epoch {epoch:3d} | val CER {vcer:.4f} | "
                  f"{time.time()-ep_t0:.1f}s/epoch")

    model.load_state_dict(best_state)                       # best-val checkpoint
    raw, refs, files = greedy_preds(model, test_dl, vocab)  # M0 predictions
    dec = lexicon_decode(raw, CFG.best_tau)                 # M2 predictions

    def split_metrics(preds):
        all_m = corpus_metrics(preds, refs)
        groups = {"seen": ([], []), "unseen": ([], [])}
        for p, r, fn in zip(preds, refs, files):
            k = "seen" if seen_map.get(fn, False) else "unseen"
            groups[k][0].append(p); groups[k][1].append(r)
        out = {"CER": all_m["CER"], "EM": all_m["ExactMatch"]}
        for k, (P, R) in groups.items():
            out[f"{k}_EM"] = corpus_metrics(P, R)["ExactMatch"] if R else float("nan")
        return out

    # per-example correctness (1/0) for paired significance testing
    m0_correct = np.array([int(p == r) for p, r in zip(raw, refs)])
    m2_correct = np.array([int(p == r) for p, r in zip(dec, refs)])
    return split_metrics(raw), split_metrics(dec), m0_correct, m2_correct

## 4. Run all seeds (resumable — completed seeds are skipped)

In [33]:
# Build data once; vocab is seed-independent (built from training labels)
_train_ds = WordImageDataset(CFG.data_root / CFG.train_csv, CFG.data_root / CFG.img_dir,
                             CFG, augment=True)
VOCAB = Vocab(_train_ds.labels()); _train_ds.vocab = VOCAB
val_ds = WordImageDataset(CFG.data_root / CFG.val_csv, CFG.data_root / CFG.img_dir, CFG, vocab=VOCAB)
test_ds = WordImageDataset(CFG.data_root / CFG.test_csv, CFG.data_root / CFG.img_dir, CFG, vocab=VOCAB)

train_dl = DataLoader(_train_ds, CFG.batch_size, shuffle=True, num_workers=CFG.num_workers,
                      collate_fn=collate, drop_last=True)
val_dl = DataLoader(val_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)
test_dl = DataLoader(test_ds, CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=collate)

test_meta = pd.read_csv(CFG.data_root / CFG.test_csv)
seen_map = dict(zip(test_meta[CFG.img_col].astype(str), test_meta["seen_in_train"]))

results_path = CFG.out_dir / "results.csv"
done = set()
if results_path.exists():
    prev = pd.read_csv(results_path)
    done = set(prev["seed"].tolist())
    print(f"resuming — already done: {sorted(done)}")

# Safety warning: if every seed is already done, NOTHING will train. This usually means a
# stale results.csv from a previous run. Delete checkpoints/multiseed/ for a true re-run.
remaining_seeds = [s for s in CFG.seeds if s not in done]
if not remaining_seeds:
    print("\n*** WARNING: all seeds already in results.csv — nothing will be trained. ***")
    print("*** If you intended a fresh run, delete the folder below and re-run this cell: ***")
    print(f"***   {CFG.out_dir.resolve()}")
else:
    print(f"seeds to train this run: {remaining_seeds}")

rows = []
overall_t0 = time.time()
seed_durations = []
for k, seed in enumerate(remaining_seeds, 1):
    t0 = time.time()
    m0, m2, c0, c2 = run_one_seed(seed, VOCAB, train_dl, val_dl, test_dl, seen_map)
    dur = time.time() - t0
    seed_durations.append(dur)
    np.savez(CFG.out_dir / f"paired_seed{seed}.npz", m0=c0, m2=c2)
    rows.append({"seed": seed,
                 "M0_CER": m0["CER"], "M0_EM": m0["EM"], "M0_seenEM": m0["seen_EM"], "M0_unseenEM": m0["unseen_EM"],
                 "M2_CER": m2["CER"], "M2_EM": m2["EM"], "M2_seenEM": m2["seen_EM"], "M2_unseenEM": m2["unseen_EM"]})
    # append to CSV immediately (crash-safe)
    df_new = pd.DataFrame(rows[-1:])
    df_new.to_csv(results_path, mode="a", header=not results_path.exists(), index=False)

    # --- timing report: this seed, running total, and estimated time remaining ---
    avg = sum(seed_durations) / len(seed_durations)
    remaining = avg * (len(remaining_seeds) - k)
    print(f"seed {seed} done in {dur/60:.1f} min "
          f"({k}/{len(remaining_seeds)}) | M0 EM {m0['EM']:.4f} -> M2 EM {m2['EM']:.4f}")
    print(f"    elapsed {((time.time()-overall_t0)/60):.1f} min | "
          f"avg {avg/60:.1f} min/seed | est. remaining {remaining/60:.1f} min")

if rows:
    print(f"\nall seeds complete. total training time: {(time.time()-overall_t0)/60:.1f} min "
          f"({(time.time()-overall_t0)/3600:.2f} h)")
else:
    print("\nno new seeds were trained (see warning above).")

print("\nall seeds complete.")

seeds to train this run: [42, 43, 44, 45, 46]
      [seed 42] epoch   1 | val CER 1.0000 | 19.3s/epoch
      [seed 42] epoch  10 | val CER 0.8381 | 18.0s/epoch
      [seed 42] epoch  20 | val CER 0.8046 | 18.0s/epoch
      [seed 42] epoch  30 | val CER 0.7725 | 17.9s/epoch
      [seed 42] epoch  40 | val CER 0.6697 | 18.0s/epoch
      [seed 42] epoch  50 | val CER 0.5918 | 18.0s/epoch
      [seed 42] epoch  60 | val CER 0.5890 | 17.9s/epoch
      [seed 42] epoch  70 | val CER 0.5300 | 17.9s/epoch
      [seed 42] epoch  80 | val CER 0.5615 | 17.9s/epoch
      [seed 42] epoch  90 | val CER 0.5116 | 18.7s/epoch
      [seed 42] epoch 100 | val CER 0.4532 | 17.9s/epoch
      [seed 42] epoch 110 | val CER 0.4434 | 17.9s/epoch
      [seed 42] epoch 120 | val CER 0.4462 | 17.9s/epoch
seed 42 done in 36.1 min (1/5) | M0 EM 0.2061 -> M2 EM 0.3515
    elapsed 36.1 min | avg 36.1 min/seed | est. remaining 144.4 min
      [seed 43] epoch   1 | val CER 1.0000 | 18.1s/epoch
      [seed 43] epoch  10 

## 5. Aggregate: mean ± std across seeds

In [34]:
res = pd.read_csv(results_path).sort_values("seed")
print("per-seed results:")
print(res.to_string(index=False))

def mstd(col):
    return f"{res[col].mean():.4f} ± {res[col].std(ddof=1):.4f}"

summary = pd.DataFrame({
    "metric": ["CER", "ExactMatch", "seen ExactMatch", "unseen ExactMatch"],
    "M0 baseline": [mstd("M0_CER"), mstd("M0_EM"), mstd("M0_seenEM"), mstd("M0_unseenEM")],
    "M2 lexicon":  [mstd("M2_CER"), mstd("M2_EM"), mstd("M2_seenEM"), mstd("M2_unseenEM")],
})
print("\nMEAN ± STD across", len(res), "seeds:")
print(summary.to_string(index=False))
summary.to_csv(CFG.out_dir / "summary.csv", index=False)

per-seed results:
 seed   M0_CER    M0_EM  M0_seenEM  M0_unseenEM   M2_CER    M2_EM  M2_seenEM  M2_unseenEM
   42 0.459280 0.206068   0.265041          0.0 0.464820 0.351454   0.452033          0.0
   43 0.484026 0.125158   0.160976          0.0 0.487350 0.294564   0.378862          0.0
   44 0.469806 0.189633   0.243902          0.0 0.470545 0.347661   0.447154          0.0
   45 0.557156 0.067004   0.086179          0.0 0.581717 0.193426   0.248780          0.0
   46 0.434164 0.213654   0.274797          0.0 0.428624 0.388116   0.499187          0.0

MEAN ± STD across 5 seeds:
           metric     M0 baseline      M2 lexicon
              CER 0.4809 ± 0.0464 0.4866 ± 0.0573
       ExactMatch 0.1603 ± 0.0627 0.3150 ± 0.0757
  seen ExactMatch 0.2062 ± 0.0807 0.4052 ± 0.0974
unseen ExactMatch 0.0000 ± 0.0000 0.0000 ± 0.0000


## 6. Significance test: does M2 beat M0?

Two complementary tests:
 1. **Paired t-test on per-seed ExactMatch** — tests whether the mean EM difference across
    seeds is non-zero. Few samples (n = #seeds), so report alongside test 2.
 2. **McNemar's test on pooled per-example outcomes** — the appropriate test for paired
    binary correct/incorrect labels on the SAME test items; far more statistical power.

Report both. p < 0.05 supports "the lexicon significantly improves exact-match accuracy".

In [35]:
# Test 1: paired t-test across seeds
t_stat, t_p = stats.ttest_rel(res["M2_EM"], res["M0_EM"])
print(f"paired t-test (per-seed EM):  t={t_stat:.3f}, p={t_p:.4g}  (n={len(res)} seeds)")

# Test 2: McNemar on pooled per-example outcomes
b = c = 0   # b: M0 right & M2 wrong; c: M0 wrong & M2 right
for seed in res["seed"]:
    d = np.load(CFG.out_dir / f"paired_seed{seed}.npz")
    m0, m2 = d["m0"], d["m2"]
    b += int(np.sum((m0 == 1) & (m2 == 0)))
    c += int(np.sum((m0 == 0) & (m2 == 1)))
# McNemar exact (binomial) test on discordant pairs
n_disc = b + c
mcnemar_p = stats.binomtest(min(b, c), n_disc, 0.5).pvalue if n_disc > 0 else 1.0
print(f"McNemar (pooled per-example): M0-only-correct={b}, M2-only-correct={c}, "
      f"discordant={n_disc}, p={mcnemar_p:.4g}")
print("  (c >> b with small p  =>  the lexicon corrects far more than it breaks, significantly)")

paired t-test (per-seed EM):  t=17.842, p=5.798e-05  (n=5 seeds)
McNemar (pooled per-example): M0-only-correct=0, M2-only-correct=612, discordant=612, p=1.177e-184
  (c >> b with small p  =>  the lexicon corrects far more than it breaks, significantly)


## 7. For the thesis
- Report the **mean ± std** table (Section 5) as your headline results table — never a
  single run again.
- Quote both significance tests (Section 6). McNemar is the statistically correct one for
  paired binary outcomes; the t-test is the intuitive per-seed version.
- The run-to-run variance you discovered is itself worth a sentence in the methodology:
  it justifies multi-seed evaluation and shows methodological maturity.
- **Unseen EM** will still be low with the training-only lexicon — the external formulary
  is the lever. State as limitation + future work.

**Next (M3):** structured {name, strength, unit} decoding + field-level / clinical metrics,
evaluated with this same multi-seed protocol.